# Proof of Concept: 2D Gaussian Splatting (2DGS)

Notebook này được thiết kế để kiểm chứng (Validate Offline) xem 2DGS có thực sự mang lại kết quả (PSNR, SSIM, LPIPS) tốt hơn Vanilla 3DGS hay không.

**Cách hoạt động:**
Chạy vòng lặp qua các scene của `public_set`. Sử dụng cờ `--eval` khi train để 2DGS tự động tách 1/8 số ảnh trong thư mục `train` ra làm tập Test (hold-out) và in ra metrics (LPIPS/SSIM/PSNR).

In [ ]:
# 1. Clone và cài đặt 2DGS
!git clone https://github.com/hbb1/2d-gaussian-splatting --recursive
%cd 2d-gaussian-splatting

# Thay thế simple-knn bằng bản fix lỗi build của camenduru
!rm -rf submodules/simple-knn
!git clone https://github.com/camenduru/simple-knn.git submodules/simple-knn

# Patch để tránh lỗi Read-only file system khi chuyển đổi points3D.bin sang points3D.ply
import os
readers_path = 'scene/dataset_readers.py'
if os.path.exists(readers_path):
    with open(readers_path, 'r') as f:
        code = f.read()
    old_code = 'ply_path = os.path.join(path, "sparse/0/points3D.ply")'
    new_code = '''ply_path = os.path.join(path, "sparse/0/points3D.ply")
    if not os.path.exists(ply_path):
        import hashlib
        path_hash = hashlib.md5(path.encode()).hexdigest()
        ply_path_tmp = os.path.join("/tmp", f"points3D_{path_hash}.ply")
        if os.path.exists(ply_path_tmp):
            ply_path = ply_path_tmp
        else:
            ply_path = ply_path_tmp'''
    if old_code in code:
        code = code.replace(old_code, new_code)
        with open(readers_path, 'w') as f:
            f.write(code)
        print('Patched dataset_readers.py successfully!')

# Patch để tránh lỗi tostring_rgb() của matplotlib 3.8+ trong 2DGS (Đã sửa lỗi scoping np)
utils_path = 'utils/general_utils.py'
if os.path.exists(utils_path):
    with open(utils_path, 'r') as f:
        code = f.read()
    old_matplotlib = 'data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)\n    data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))'
    new_matplotlib = '''try:
        data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    except AttributeError:
        data = np.asarray(fig.canvas.buffer_rgba())[..., :3]'''
    if old_matplotlib in code:
        code = code.replace(old_matplotlib, new_matplotlib)
        with open(utils_path, 'w') as f:
            f.write(code)
        print('Patched general_utils.py for matplotlib compatibility successfully!')
    else:
        old_matplotlib_indented = '    data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)\n    data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))'
        new_matplotlib_indented = '''    try:
        data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    except AttributeError:
        data = np.asarray(fig.canvas.buffer_rgba())[..., :3]'''
        if old_matplotlib_indented in code:
            code = code.replace(old_matplotlib_indented, new_matplotlib_indented)
            with open(utils_path, 'w') as f:
                f.write(code)
            print('Patched general_utils.py (indented) successfully!')

!pip install -q plyfile tqdm opencv-python joblib pillow lpips
!pip install -q submodules/diff-surfel-rasterization
!pip install -q submodules/simple-knn

In [ ]:
import os
import subprocess

# 2. Cấu hình đường dẫn và scene cần test
DATASET_ROOT = '/kaggle/input/datasets/ptquanh/vtar-b1-preprocessed-dataset/public_set'
OUTPUT_ROOT = '/kaggle/working/outputs_2dgs'

# Danh sách scene cần test. Để None hoặc [] nếu muốn train TẤT CẢ các scene trong folder
TARGET_SCENES = ['HCM0204']  # Ví dụ: ['HCM0204']

# Các tham số training
ITERATIONS = 15000

if not os.path.exists(DATASET_ROOT):
    print(f"Thư mục {DATASET_ROOT} không tồn tại. Vui lòng kiểm tra lại data input!")
else:
    all_scenes = sorted(os.listdir(DATASET_ROOT))
    
    # Lọc scene theo TARGET_SCENES
    if TARGET_SCENES:
        scenes = [s for s in all_scenes if s in TARGET_SCENES]
    else:
        scenes = all_scenes
        
    print(f"Tìm thấy {len(all_scenes)} scenes trong dataset.")
    print(f"Sẽ thực hiện train {len(scenes)} scenes để test 2DGS: {scenes}")
    
    for scene in scenes:
        print("\n" + "="*60)
        print(f"TRAINING SCENE: {scene} WITH 2DGS")
        print("="*60)
        
        scene_path = os.path.join(DATASET_ROOT, scene, 'train')
        output_dir = os.path.join(OUTPUT_ROOT, scene)
        
        command = [
            "python", "train.py",
            "-s", scene_path,
            "-m", output_dir,
            "--eval",
            "--iterations", str(ITERATIONS),
            "--test_iterations", "7000", str(ITERATIONS),
            "--save_iterations", str(ITERATIONS),
            "--checkpoint_iterations", str(ITERATIONS),
            "--position_lr_max_steps", str(ITERATIONS),
        ]
        
        # Chạy lệnh train và in log trực tiếp
        subprocess.run(command, check=True)
        print(f"Hoàn thành train 2DGS cho scene {scene}!")

## Đánh giá kết quả
Sau khi mỗi scene chạy xong, cuộn lên phần log của `train.py`. Bạn sẽ thấy log dạng:
```
[ITER 15000] Evaluating test: L1 ... PSNR ... SSIM ... LPIPS ...
```
Ghi lại 3 con số PSNR, SSIM, LPIPS này để so sánh với Vanilla 3DGS.